# Epi Info AI Rates validation lab — V0.11

Validate the candidate Visual Dashboard `epi.rate` Rust/WebAssembly kernel against the canonical foodborne data and an independent Python calculation. Passing is evidence, not statistical approval; G5 remains consolidated.

In [ ]:
import csv, hashlib, io, math
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
fixture_response = await pyfetch('../../validation-fixtures/foodborne-rate-v0.11.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
data_response = await pyfetch('../../examples/foodborne-outbreak-investigation.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert hashlib.sha256(data_bytes).hexdigest() == fixture['dataset']['sha256']
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
wasm_response = await pyfetch('../../epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports


In [ ]:
request = fixture['request']; eligible = [row for row in records if row[request['denominatorSourceHeader']].strip()]
numerator = sum(row[request['numeratorSourceHeader']].strip().casefold() == request['numeratorValue'].casefold() for row in eligible)
denominator = len(eligible); independent_rate = numerator / denominator * request['multiplier']
assert numerator == fixture['expected']['numerator']; assert denominator == fixture['expected']['denominator']
assert math.isclose(independent_rate, fixture['expected']['rate'], abs_tol=1e-12, rel_tol=0)
print('PASS: independent Python derivation matches the foodborne V0.11 anchor')


In [ ]:
candidate = float(rust.rate_calculate(numerator, denominator, request['multiplier']))
assert math.isclose(candidate, independent_rate, abs_tol=1e-12, rel_tol=0)
assert math.isnan(float(rust.rate_calculate(1, 0, 100)))
print(f'PASS: deployed Rust/WASM rate = {candidate:.12f} per {request["multiplier"]:,}')
